In [1]:
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import os
from tqdm import tqdm
import pandas as pd
import gc
import torch

In [2]:
def extract_frames(video_path, skip_frames=5):
    """
    Args:
        video_path: video URL or path
        skip_frames: exctract only one of N frames
    
    Returns:
        tuple (delta_t, frames):
            delta_t: time between frames exctracted
            frames: np.array of frames
    """
    cap = cv2.VideoCapture(video_path)
    
    frames = []
    frame_count = 0
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    delta_t = skip_frames / cap.get(cv2.CAP_PROP_FPS)
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % skip_frames == 0:
            frames.append(frame)
        
        frame_count += 1
    
    cap.release()
    #print(f"Frame count: {frame_count}")
    #print(f"Frames appended: {len(frames)} (each {skip_frames})")
    
    return delta_t, frames

def split_into_segments(frames, segment_length=10):
    """
    Args:
        frames: array of frames
        segment_length: how much frames in one split
    
    Returns:
        array of segments, each contains np.array of frames
    """
    segments = []
    num_segments = len(frames) // segment_length
    
    for i in range(num_segments):
        start = i * segment_length
        end = start + segment_length
        segment = frames[start:end]
        segments.append(segment)
        
    remainder = len(frames) % segment_length
    
    print(f"Segment count: {len(segments)}")
    return segments

In [3]:
import albumentations as A

def augment_segment(frames):
    transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussianBlur(p=0.3),
    ])
    
    augmented = transform(images=frames)
    aug_images = augmented["images"]
    return aug_images

In [4]:
landmark_names = [
    'nose',
    'left_eye_inner', 'left_eye', 'left_eye_outer',
    'right_eye_inner', 'right_eye', 'right_eye_outer',
    'left_ear', 'right_ear',
    'mouth_left', 'mouth_right',
    'left_shoulder', 'right_shoulder',
    'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist',
    'left_pinky_1', 'right_pinky_1',
    'left_index_1', 'right_index_1',
    'left_thumb_2', 'right_thumb_2',
    'left_hip', 'right_hip',
    'left_knee', 'right_knee',
    'left_ankle', 'right_ankle',
    'left_heel', 'right_heel',
    'left_foot_index', 'right_foot_index',
]

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
model_path = "mediapipe/pose_landmarker_full.task"

def extract_pose_from_segment(segment, delta_t, label, poses=1):
    """
    Returns:
        dataframe (N, 33*3)
    """    
    all_landmarks = []
    
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=poses,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5
    )
    pose_landmarker = PoseLandmarker.create_from_options(options)
    
    for i, frame in enumerate(segment):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        frame_timestamp_ms = int(i * delta_t * 1000)
        detection_result = pose_landmarker.detect_for_video(mp_image, frame_timestamp_ms)

        if detection_result.pose_landmarks and len(detection_result.pose_landmarks) > 0:
            pose_landmarks = detection_result.pose_landmarks[0]

            landmarks = []
            for landmark in pose_landmarks:
                landmarks.append([landmark.x, landmark.y, landmark.z])
            all_landmarks.append(landmarks)
        else:
            #0 if no person
            all_landmarks.append([[0,0,0] for _ in range(33)])
    
    videodata = torch.tensor(all_landmarks)
    
    return videodata

In [5]:
def get_labels_from_folders(data_path):
    labels = os.listdir(data_path)
    print(f"Found {len(labels)} labels:")
    print(labels)
    return labels

def get_video_paths(data_path, label):
    label_path = data_path+label
    videos = os.listdir(label_path)
    return videos

def process_video_dataset(labels_dir_path, save_dir, skip_frames=3, segment_length=10):
    """
    Process each video of the dataset
    
    Args:
        labels_dir_path: path to every label directory
        skip_frames: process only nth frame
        segment_length: amount of frames in a segment
    
    Returns:
        Tensor with shape(frames, pose_landmarks=33, dimension=3)
    """
    
    labels = get_labels_from_folders(labels_dir_path)
    
    for label in tqdm(labels, desc="Labels"):
        videos = get_video_paths(labels_dir_path, label)
        
        for index, video in enumerate(tqdm(videos, desc=f"Videos ({label})")):

            video_path =  labels_dir_path+label+"/"+video
            delta_t, frames = extract_frames(video_path, skip_frames)
            segments = split_into_segments(frames, segment_length)
            
            if len(segments) == 0:
                print(f"Warning: video {video_path} too short, skipping")
                continue
            
            all_segments_tensors = []
            
            for segment in segments:
                data_tens = extract_pose_from_segment(segment, delta_t, label)
                
                all_segments_tensors.append(data_tens)
            
            if len(all_segments_tensors) != 0:
                video_tensor = torch.stack(all_segments_tensors)
                video_tensor = video_tensor.permute(0, 3, 1, 2) # (number, 3, frames, 33)
                
                torch.save(video_tensor, f"{save_dir}/{label}-{index}.pt")
                del all_segments_tensors, video_tensor
                gc.collect()
            else:
                print(f"Warning: No segments extracted from video {video_path}")

In [6]:
data_path_train = "data/verified_data/"
test_path_test = "data/test/test/"
train_btc = "data_btc_10s/"
train_crawl = "data_crawl_10s/"
save_dir = "data/dataCNNdf/verified_data/"
save_dir_test = "data/dataCNNdf/test/"

In [7]:
#test
test_video = "data/verified_data/data_btc_10s/hip thrust/58ad2615-4c71-40c3-ad0e-fdc3ef4a331f.mp4"
delta_t, test_frames = extract_frames(test_video, skip_frames=4)
print(f"Frames extracted: {len(test_frames)}")

Frames extracted: 30


In [8]:
test_segments = split_into_segments(test_frames)

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5
)
pose_landmarker = PoseLandmarker.create_from_options(options)

df = extract_pose_from_segment(test_segments[0], delta_t, 'hip thrust')
df.shape

Segment count: 3


torch.Size([10, 33, 3])

In [9]:
process_video_dataset(data_path_train+train_btc, save_dir)

Found 22 labels:
['barbell biceps curl', 'bench press', 'chest fly machine', 'deadlift', 'decline bench press', 'hammer curl', 'hip thrust', 'incline bench press', 'lat pulldown', 'lateral raise', 'leg extension', 'leg raises', 'plank', 'pull Up', 'push-up', 'romanian deadlift', 'russian twist', 'shoulder press', 'squat', 't bar row', 'tricep dips', 'tricep Pushdown']


Videos (barbell biceps curl):   0%|                                                             | 0/63 [00:00<?, ?it/s]

Segment count: 2



Videos (barbell biceps curl):   2%|▊                                                    | 1/63 [00:01<01:09,  1.13s/it]

Segment count: 7


...


Videos (tricep Pushdown):  96%|█████████████████████████████████████████████████████▊  | 48/50 [01:19<00:04,  2.26s/it]

Segment count: 3



Videos (tricep Pushdown):  98%|██████████████████████████████████████████████████████▉ | 49/50 [01:20<00:02,  2.07s/it]

Segment count: 2



Labels: 100%|██████████████████████████████████████████████████████████████████████████| 22/22 [36:11<00:00, 98.72s/it]


In [10]:
save_dir = "data/dataCNNdf/verified_data_crawl/"
process_video_dataset(data_path_train+train_crawl, save_dir)

Found 22 labels:
['barbell biceps curl', 'bench press', 'chest fly machine', 'deadlift', 'decline bench press', 'hammer curl', 'hip thrust', 'incline bench press', 'lat pulldown', 'lateral raise', 'leg extension', 'leg raises', 'plank', 'pull Up', 'push-up', 'romanian deadlift', 'russian twist', 'shoulder press', 'squat', 't bar row', 'tricep dips', 'tricep Pushdown']


Videos (barbell biceps curl):   0%|                                                             | 0/48 [00:00<?, ?it/s]

Segment count: 9



Videos (barbell biceps curl):   2%|█                                                    | 1/48 [00:04<03:45,  4.80s/it]

Segment count: 8


...


Videos (tricep Pushdown):  88%|█████████████████████████████████████████████████       | 14/16 [00:57<00:07,  3.86s/it]

Segment count: 8



Videos (tricep Pushdown):  94%|████████████████████████████████████████████████████▌   | 15/16 [01:00<00:03,  3.63s/it]

Segment count: 8



Labels: 100%|█████████████████████████████████████████████████████████████████████████| 22/22 [55:19<00:00, 150.89s/it]


In [11]:
process_video_dataset(test_path_test, save_dir)

Found 22 labels:
['barbell biceps curl', 'bench press', 'chest fly machine', 'deadlift', 'decline bench press', 'hammer curl', 'hip thrust', 'incline bench press', 'lat pulldown', 'lateral raise', 'leg extension', 'leg raises', 'plank', 'pull Up', 'push-up', 'romanian deadlift', 'russian twist', 'shoulder press', 'squat', 't bar row', 'tricep dips', 'tricep Pushdown']


Videos (barbell biceps curl):   0%|                                                              | 0/2 [00:00<?, ?it/s]

Segment count: 26



Videos (barbell biceps curl):  50%|███████████████████████████                           | 1/2 [00:08<00:08,  8.53s/it]

Segment count: 14


...


Videos (tricep Pushdown):   0%|                                                                  | 0/2 [00:00<?, ?it/s]

Segment count: 14



Videos (tricep Pushdown):  50%|█████████████████████████████                             | 1/2 [00:05<00:05,  5.07s/it]

Segment count: 9



Labels: 100%|██████████████████████████████████████████████████████████████████████████| 22/22 [07:22<00:00, 20.12s/it]


In [14]:
test_tens = torch.load('data/dataCNNdf/verified_data/barbell biceps curl-10.pt')
test_tens.shape

torch.Size([20, 3, 10, 33])